In [5]:
import getpass
import os
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain.agents import create_agent
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model
from pathlib import Path
import pandas as pd

# Configurar API key
if not os.environ.get("DEEPSEEK_API_KEY"):
    os.environ["DEEPSEEK_API_KEY"] = getpass.getpass("Enter API key for DeepSeek: ")

# Inicializar o modelo DeepSeek
llm = init_chat_model("deepseek-chat", model_provider="deepseek", temperature=0)


In [34]:
df = pd.read_csv(Path.cwd().parent / 'data' / '20260115_1403_contos_metadados_basicos.csv')
df.head()


,Unnamed: 0.1,Unnamed: 0,url,title,full_text_EN,region,author,clean_text_EN,source,word_count,reading_time,flesch_reading_ease,dale_chall_readability
0,0,0,https://sites.pitt.edu/~dash/manykids.html,The Birth of Aistulf,The following legend is told about King Aistul...,Germany,Jacob and Wilhelm Grimm,The following legend is told about King Aistul...,"Source: Jacob and Wilhelm Grimm, ""Aistulfs Geb...",81,0.405,85.947556,6.779279
1,1,1,https://sites.pitt.edu/~dash/manykids.html,As Many Children as There Are Days in the Year,Loosduynen (Leusden) is a small village one mi...,Germany,Jacob and Wilhelm Grimm,Loosduynen (Leusden) is a small village one mi...,"Source: Jacob and Wilhelm Grimm, ""So viel Kind...",250,1.250,72.300400,9.448694
2,2,2,https://sites.pitt.edu/~dash/manykids.html,The Woman with Three Hundred and Sixty-Six \nC...,"So, because the forests of oak, and beech, and...",Netherlands,NaN,"So, because the forests of oak, and beech, and...","Source: William Elliot Griffis,Dutch Fairy Tal...",2709,13.545,75.547294,7.532870
3,3,3,https://sites.pitt.edu/~dash/manykids.html,The Boy in the Fishpond,"In the times of Agelmund, the King of the Lang...",Germany,Jacob and Wilhelm Grimm,"In the times of Agelmund, the King of the Lang...","Source: Jacob and Wilhelm Grimm, ""Der Knabe i...",168,0.840,80.508000,7.383411
4,4,4,https://sites.pitt.edu/~dash/manykids.html,The Origin of the Welfs,Warin was a count of Altorf and Ravensburg in ...,Germany,Jacob and Wilhelm Grimm,Warin was a count of Altorf and Ravensburg in ...,"Source: Jacob and Wilhelm Grimm, ""Ursprung de...",589,2.945,72.175178,8.155761


In [33]:
# df = df.tail(100)
# print(df.shape)
# df.head()

In [14]:
df.columns

Index(['Unnamed: 0.1', 'Unnamed: 0', 'url', 'title', 'full_text_EN', 'region',
       'author', 'clean_text_EN', 'source', 'word_count', 'reading_time',
       'flesch_reading_ease', 'dale_chall_readability'],
      dtype='object')

In [ ]:
from pydantic import BaseModel, Field
from typing import List
from langchain_core.prompts import ChatPromptTemplate
import pandas as pd
from time import sleep

# Define structured output schema matching your categories
class StoryClassifications(BaseModel):
    """Extract classifications from story text for media/SEO."""
    image_prompt: str = Field(description="Vivid scene description for Midjourney/DALL-E cover image (50-100 words).")
    voice_profile: str = Field(description="TTS voice suggestion like 'Old wise man', 'Cheerful child'.")
    mood: str = Field(description="Primary feeling for soundtrack: Terror, Epic, Melancholic, Joyful, etc.")
    tags: List[str] = Field(description="5-10 thematic keywords for SEO, comma-separated.")
    moral: str = Field(description="Core lesson/moral of the story (1 sentence, school-friendly).")
    entities: List[str] = Field(description="Mythical creatures mentioned: Fadas, Goblins, Dragões, etc.")

# Bind schema to your DeepSeek model (uses tool calling for structured output)
structured_llm = llm.with_structured_output(StoryClassifications)

# Prompt template for context
prompt = ChatPromptTemplate.from_messages([
    ("system", """Analyze this story text and extract ONLY the classifications in the exact schema format.
    Use `clean_text_EN` primarily, fall back to `title` if needed. Be precise and concise."""),
    ("human", "{text}")
])

chain = prompt | structured_llm

def classify_row(row, index):
    text = row.get('clean_text_EN', '') or row.get('full_text_EN', '') or row['title']
    if not text.strip():
        df.at[index, 'classification_cost_usd'] = 0.0
        df.at[index, 'input_tokens'] = 0
        df.at[index, 'output_tokens'] = 0
        return {f: "" for f in StoryClassifications.model_fields}, 0.0
    
    result = chain.invoke({"text": text[:4000]})
    
    # Inicializar custo como 0
    total_cost_usd = 0.0
    
    # Extrair tokens da resposta (AIMessage ou structured output)
    if hasattr(result, 'usage_metadata') and result.usage_metadata:
        usage = result.usage_metadata
        input_tokens = usage.get('input_tokens', 0)
        output_tokens = usage.get('output_tokens', 0)
        
        df.at[index, 'input_tokens'] = input_tokens
        df.at[index, 'output_tokens'] = output_tokens
        
        # Preços DeepSeek deepseek-chat (confirme em platform.deepseek.com/pricing)
        input_cost = input_tokens / 1e6 * 0.14
        output_cost = output_tokens / 1e6 * 0.28
        total_cost_usd = input_cost + output_cost
        
        df.at[index, 'classification_cost_usd'] = round(total_cost_usd, 6)
    else:
        # Fallback se não houver usage_metadata
        df.at[index, 'classification_cost_usd'] = 0.0
        df.at[index, 'input_tokens'] = 0
        df.at[index, 'output_tokens'] = 0
    
    return result.dict(), total_cost_usd

# Inicializar colunas de custo
df['classification_cost_usd'] = 0.0
df['input_tokens'] = 0
df['output_tokens'] = 0

# Aplicar com index
results = []
total_cost = 0.0
for idx, row in df.iterrows():
    classifications, cost = classify_row(row, idx)
    results.append(classifications)
    total_cost += cost
    sleep(0.1)

print(f"Custo total: ${total_cost:.6f}")
display(df[['title', 'classification_cost_usd', 'input_tokens', 'output_tokens']].head())

/tmp/ipykernel_474766/962452951.py:63: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  return result.dict(), total_cost_usd


In [ ]:
df.to_csv(Path.cwd().parent / 'data' / '20260116_1730_contos_metadados_basicos.csv', index=False)

,Unnamed: 0.1,Unnamed: 0,url,title,full_text_EN,region,author,clean_text_EN,source,word_count,reading_time,flesch_reading_ease,dale_chall_readability,classification_cost_usd,input_tokens,output_tokens
1303,1303,1303,https://sites.pitt.edu/~dash/type1319.html,Kleinenberg Horse Eggs,The people of Kleinenberg were curious when th...,Germany,NaN,The people of Kleinenberg were curious when th...,"Source (books.google.com): Adalbert Kuhn, ""Kle...",157,0.785,87.978481,6.929551,0.0,0,0
1304,1304,1304,https://sites.pitt.edu/~dash/type1319.html,The Men from Ried Hatch a Donkey,Now the people of Ried were a simple and backw...,Switzerland,NaN,Now the people of Ried were a simple and backw...,Source (books.google.com): Johannes Jegerlehne...,208,1.040,83.898397,6.697724,0.0,0,0
1305,1305,1305,https://sites.pitt.edu/~dash/type1319.html,Donkey Seed,"The bishop politely listened to them, then sai...",Italy,NaN,"The bishop politely listened to them, then sai...",Source (books.google.com): Christian Schneller...,112,0.560,85.432460,7.214369,0.0,0,0
1306,1306,1306,https://sites.pitt.edu/~dash/type1319.html,The Pumpkin,"They all stood there baffled, until finally on...",Serbia,NaN,"They all stood there baffled, until finally on...",Source (books.google.com): Friedrich S. Krauss...,276,1.380,75.665522,7.066386,0.0,0,0
1307,1307,1307,https://sites.pitt.edu/~dash/type1319.html,The People of Sainte-Dode,One day it occurred to the people of Sainte-Do...,France,NaN,One day it occurred to the people of Sainte-Do...,"Source (books.google.com): E. K. Blümml, ""Die ...",368,1.840,85.893151,7.094319,0.0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1398,1398,1398,https://sites.pitt.edu/~dash/jataka.html,The Timid Hare and the Flight of the Beasts,Once upon a time when Brahmadatta reigned in B...,Unknown,NaN,Once upon a time when Brahmadatta reigned in B...,NaN,972,4.860,85.085631,7.172228,0.0,0,0
1399,1399,1399,https://sites.pitt.edu/~dash/jataka.html,How a Vain Woman Was Reborn As a Dung-Worm,She died; and at her death the king was plunge...,Unknown,NaN,She died; and at her death the king was plunge...,"Source:The Jataka; or, Stories of the Buddha's...",785,3.925,80.135711,7.733637,0.0,0,0
1400,1400,1400,https://sites.pitt.edu/~dash/jataka.html,The Language of Animals,Once upon a time when a king named Senaka was ...,Unknown,NaN,Once upon a time when a king named Senaka was ...,NaN,1498,7.490,86.953645,6.956067,0.0,0,0
1401,1401,1401,https://sites.pitt.edu/~dash/jataka.html,Sulasa and Sattuka,Once upon a time when Brahmadatta was reigning...,Unknown,NaN,Once upon a time when Brahmadatta was reigning...,NaN,540,2.700,81.508333,6.887220,0.0,0,0


In [1]:
df

NameError: name 'df' is not defined